In [ ]:
import subprocess
import sys

print("=" * 60)
print("Checking and installing required packages...")
print("=" * 60)

required_packages = [
    'transformers>=4.36.0',
    'torch',
    'accelerate',
    'pandas',
    'scikit-learn',
    'huggingface_hub',
    'tqdm',
    'scipy',
    'numpy',
    'joblib'
]

for package in required_packages:
    package_name = package.split('>=')[0] if '>=' in package else package
    try:
        __import__(package_name)
        print(f"✓ {package_name} is already installed")
    except ImportError:
        print(f"✗ Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, f1_score
from transformers import AutoTokenizer, AutoModel
import numpy as np
import gc
import warnings
import os
from tqdm import tqdm
from huggingface_hub import login, HfApi
import random
from scipy import stats
import joblib
from collections import Counter
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def load_and_prepare_authorship_dataset(file_path, test_size=0.2, min_samples_per_author=2, max_authors=50, seed=42):
    """
    Load Google Jam dataset for authorship attribution
    """
    print("Loading dataset...")
    data = pd.read_csv(file_path)
    
    data = data.dropna(subset=['flines', 'username'])
    data['flines'] = data['flines'].astype(str)
    data = data[data['flines'].str.strip() != '']
    
    author_counts = data['username'].value_counts()
    
    valid_authors = author_counts[author_counts >= min_samples_per_author].index
    filtered_data = data[data['username'].isin(valid_authors)]
    
    if len(valid_authors) > max_authors:
        top_authors = author_counts.head(max_authors).index
        filtered_data = filtered_data[filtered_data['username'].isin(top_authors)]
    
    label_encoder = LabelEncoder()
    filtered_data['EncodedLabels'] = label_encoder.fit_transform(filtered_data['username'])
    num_classes = len(label_encoder.classes_)
    
    train_data, test_data = custom_stratified_split(filtered_data, test_size=test_size, seed=seed)
    
    print(f"\nDataset Statistics:")
    print(f"  Number of authors: {num_classes}")
    print(f"  Training samples: {len(train_data)}")
    print(f"  Test samples: {len(test_data)}")
    print(f"  Total samples: {len(train_data) + len(test_data)}")
    
    print(f"\nTop 5 authors sample counts:")
    for i, author in enumerate(label_encoder.classes_[:5]):
        train_count = len(train_data[train_data['username'] == author])
        test_count = len(test_data[test_data['username'] == author])
        print(f"  {i+1}. {author}: Train={train_count}, Test={test_count}")
    
    return train_data, test_data, label_encoder, num_classes

def custom_stratified_split(data, test_size=0.2, seed=42):
    """Custom stratified split that ensures proper distribution"""
    train_data = []
    test_data = []
    
    grouped = data.groupby('username')
    
    for author, group in grouped:
        group = group.sample(frac=1, random_state=seed).reset_index(drop=True)
        
        n_test = max(1, int(len(group) * test_size))
        
        if len(group) - n_test < 2:
            n_test = max(1, len(group) - 2)
        
        test_samples = group.iloc[:n_test]
        train_samples = group.iloc[n_test:]
        
        test_data.append(test_samples)
        train_data.append(train_samples)
    
    train_data = pd.concat(train_data, ignore_index=True)
    test_data = pd.concat(test_data, ignore_index=True)
    
    train_data = train_data.sample(frac=1, random_state=seed).reset_index(drop=True)
    test_data = test_data.sample(frac=1, random_state=seed).reset_index(drop=True)
    
    return train_data, test_data

print("\n" + "=" * 60)
print("Hugging Face Authentication")
print("=" * 60)

token = None
token_path = os.path.expanduser("~/.huggingface/token")

if os.path.exists(token_path):
    with open(token_path, 'r') as f:
        token = f.read().strip()
    print("✓ Using existing Hugging Face token")
else:
    print("\nPlease enter your Hugging Face access token.")
    print("Get it from: https://huggingface.co/settings/tokens")
    token = input("Enter token: ").strip()
    
    os.makedirs(os.path.dirname(token_path), exist_ok=True)
    with open(token_path, 'w') as f:
        f.write(token)

try:
    login(token=token)
    print("✓ Logged in to Hugging Face")
except Exception as e:
    print(f"✗ Login error: {e}")
    exit(1)

class CodeLlamaForAuthorship(nn.Module):
    """
    CodeLlama model adapted for authorship attribution
    Freezes the base model and trains only the classification head
    """
    def __init__(self, model_name="codellama/CodeLlama-7b-hf", num_classes=50, token=None):
        super(CodeLlamaForAuthorship, self).__init__()
        
        print(f"Loading {model_name}...")
        
        
        self.encoder = AutoModel.from_pretrained(
            model_name,
            dtype=torch.float16,
            token=token,
            trust_remote_code=True,
            device_map="auto" if torch.cuda.is_available() else None
        )
        
        print("✓ Model loaded successfully!")
        
       
        self.hidden_size = self.encoder.config.hidden_size
        print(f"  Hidden size: {self.hidden_size}")
        
       
        for param in self.encoder.parameters():
            param.requires_grad = False
        print("  Base model frozen")
        
        self.classifier = nn.Sequential(
            nn.Linear(self.hidden_size, 1024),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )
        
        for layer in self.classifier:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)
        
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.classifier.parameters())
        
        print(f"  Total parameters: {total_params:,}")
        print(f"  Trainable parameters: {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")
    
    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
        
        last_hidden_state = outputs.last_hidden_state
        
        attention_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * attention_mask_expanded, dim=1)
        sum_mask = torch.clamp(attention_mask_expanded.sum(dim=1), min=1e-9)
        pooled_output = sum_embeddings / sum_mask
        
        logits = self.classifier(pooled_output)
        return logits

def run_codellama_experiment(num_seeds=10, model_size="7b", num_epochs=6):
    """
    Run CodeLlama experiment across multiple seeds
    """
    print("\n" + "="*80)
    print(f"CodeLlama-{model_size} AUTHORSHIP ATTRIBUTION EXPERIMENT")
    print("="*80)
    
    model_names = {
        "7b": "codellama/CodeLlama-7b-hf",
        "13b": "codellama/CodeLlama-13b-hf",
        "34b": "codellama/CodeLlama-34b-hf"
    }
    
    if model_size not in model_names:
        print(f"Warning: Model size {model_size} not recognized. Using 7b.")
        model_size = "7b"
    
    model_name = model_names[model_size]
    print(f"Using model: {model_name}")
    
    SEEDS = [42, 123, 456, 789, 999, 111, 222, 333, 444, 555][:num_seeds]
    
    file_path = "/home/aman_swaraj/Downloads/Codelite/Other SE Tasks/Authorship_Attribution/gcj2020.csv"
    print("\nLoading dataset...")
    
    all_results = []
    all_predictions = {}
    
    for seed_idx, seed in enumerate(SEEDS):
        print(f"\n{'='*60}")
        print(f"SEED {seed_idx+1}/{num_seeds}: {seed}")
        print(f"{'='*60}")
        
        set_seed(seed)
        
        print("\n1. Loading and preparing data...")
        
        train_data, test_data, label_encoder, num_classes = load_and_prepare_authorship_dataset(
            file_path,
            test_size=0.2,
            min_samples_per_author=2,  
            max_authors=1000,  
            seed=seed
        )
        
        print("\n2. Loading tokenizer and tokenizing data...")
        
        try:
            tokenizer = AutoTokenizer.from_pretrained(
                model_name,
                token=token,
                trust_remote_code=True
            )
            
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            
            tokenizer.padding_side = "left"
            print("✓ Tokenizer loaded")
            
        except Exception as e:
            print(f"✗ Tokenizer error: {e}")
            continue
        
        def tokenize_batch(texts, max_length=256):
            """Efficient batch tokenization"""
            return tokenizer(
                texts,
                max_length=max_length,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )
        
        print("  Tokenizing training data...")
        train_texts = train_data["flines"].tolist()
        train_tokenized = tokenize_batch(train_texts, max_length=256)
        
        print("  Tokenizing test data...")
        test_texts = test_data["flines"].tolist()
        test_tokenized = tokenize_batch(test_texts, max_length=256)
        
        print("\n3. Creating datasets...")
        
        class AuthorshipDataset(Dataset):
            def __init__(self, input_ids, attention_mask, labels):
                self.input_ids = input_ids
                self.attention_mask = attention_mask
                self.labels = labels
            
            def __len__(self):
                return len(self.labels)
            
            def __getitem__(self, idx):
                return {
                    "input_ids": self.input_ids[idx],
                    "attention_mask": self.attention_mask[idx],
                    "labels": self.labels[idx]
                }
        
        train_dataset = AuthorshipDataset(
            train_tokenized["input_ids"],
            train_tokenized["attention_mask"],
            torch.tensor(train_data["EncodedLabels"].values, dtype=torch.long)
        )
        
        test_dataset = AuthorshipDataset(
            test_tokenized["input_ids"],
            test_tokenized["attention_mask"],
            torch.tensor(test_data["EncodedLabels"].values, dtype=torch.long)
        )
        
        print(f"  Training set: {len(train_dataset)} samples")
        print(f"  Test set: {len(test_dataset)} samples")
        
        print("\n4. Setting up model and training...")
        
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"  Device: {device}")
        if torch.cuda.is_available():
            print(f"  GPU: {torch.cuda.get_device_name(0)}")
            print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
        
        model = CodeLlamaForAuthorship(model_name=model_name, num_classes=num_classes, token=token)
        model = model.to(device)
        
        optimizer = optim.AdamW(
            model.classifier.parameters(),
            lr=5e-5,
            weight_decay=0.01
        )
        
        criterion = nn.CrossEntropyLoss()
        
        if model_size in ["13b", "34b"]:
            batch_size = 4
        else:
            batch_size = 8
        
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
            pin_memory=True
        )
        
        test_loader = DataLoader(
            test_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0
        )
        
        print(f"\n  Training configuration:")
        print(f"    Batch size: {batch_size}")
        print(f"    Learning rate: {5e-5}")
        print(f"    Epochs: {num_epochs}")
        print(f"    Training samples: {len(train_dataset)}")
        print(f"    Test samples: {len(test_dataset)}")
        print(f"    Number of authors: {num_classes}")
        
        print("\n5. Training model...")
        
        epochs = num_epochs
        model.train()
        
        best_accuracy = 0
        train_accuracies = []
        train_losses = []
        
        for epoch in range(epochs):
            print(f"\n  Epoch {epoch+1}/{epochs}")
            print("  " + "-" * 40)
            
            total_loss = 0
            correct = 0
            total = 0
            
            progress_bar = tqdm(train_loader, desc=f"    Training")
            
            for batch_idx, batch in enumerate(progress_bar):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)
                
                optimizer.zero_grad()
                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels)
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.classifier.parameters(), max_norm=1.0)
                optimizer.step()
                
                total_loss += loss.item()
                predictions = torch.argmax(logits, dim=1)
                correct += (predictions == labels).sum().item()
                total += labels.size(0)
                
                avg_loss = total_loss / (batch_idx + 1)
                accuracy = correct / total
                progress_bar.set_postfix({
                    'loss': f'{avg_loss:.4f}',
                    'acc': f'{accuracy:.4f}'
                })
            
            avg_epoch_loss = total_loss / len(train_loader)
            epoch_accuracy = correct / total
            
            train_losses.append(avg_epoch_loss)
            train_accuracies.append(epoch_accuracy)
            
            print(f"  Epoch {epoch+1} Summary:")
            print(f"    Loss: {avg_epoch_loss:.4f}")
            print(f"    Train Accuracy: {epoch_accuracy:.4f}")
            
            model.eval()
            test_correct = 0
            test_total = 0
            
            with torch.no_grad():
                for batch in test_loader:
                    input_ids = batch["input_ids"].to(device)
                    attention_mask = batch["attention_mask"].to(device)
                    labels = batch["labels"].to(device)
                    
                    logits = model(input_ids, attention_mask)
                    predictions = torch.argmax(logits, dim=1)
                    test_correct += (predictions == labels).sum().item()
                    test_total += labels.size(0)
            
            test_accuracy = test_correct / test_total
            print(f"    Test Accuracy: {test_accuracy:.4f}")
            
            if test_accuracy > best_accuracy:
                best_accuracy = test_accuracy
                best_model_path = f"/home/aman_swaraj/Downloads/Codelite/codellama_{model_size}_seed{seed}_best.pth"
                
                torch.save(model.state_dict(), best_model_path)
                
                label_encoder_path = f"/home/aman_swaraj/Downloads/Codelite/codellama_{model_size}_seed{seed}_label_encoder.pkl"
                joblib.dump(label_encoder, label_encoder_path)
                
                metadata_path = f"/home/aman_swaraj/Downloads/Codelite/codellama_{model_size}_seed{seed}_metadata.pkl"
                metadata = {
                    'seed': seed,
                    'num_classes': num_classes,
                    'class_names': label_encoder.classes_.tolist()
                }
                joblib.dump(metadata, metadata_path)
                
                print(f"    ✓ Saved best model (accuracy: {best_accuracy:.4f})")
            
            model.train()
        
        model.load_state_dict(torch.load(best_model_path, weights_only=False))
        
        label_encoder = joblib.load(label_encoder_path)
        
        print("\n6. Final evaluation...")
        
        model.eval()
        all_preds = []
        all_labels = []
        all_confidences = []
        all_probs = []
        
        with torch.no_grad():
            for batch in tqdm(test_loader, desc="    Evaluating"):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].cpu().numpy()
                
                logits = model(input_ids, attention_mask)
                probs = torch.softmax(logits, dim=1)
                preds = torch.argmax(logits, dim=1).cpu().numpy()
                confidences = torch.max(probs, dim=1)[0].cpu().numpy()
                
                all_preds.extend(preds)
                all_labels.extend(labels)
                all_confidences.extend(confidences)
                all_probs.append(probs.cpu().numpy())
        
        test_accuracy = accuracy_score(all_labels, all_preds)
        macro_f1 = f1_score(all_labels, all_preds, average='macro')
        weighted_f1 = f1_score(all_labels, all_preds, average='weighted')
        
        seed_results = {
            'seed': seed,
            'model_size': model_size,
            'test_accuracy': test_accuracy,
            'macro_f1': macro_f1,
            'weighted_f1': weighted_f1,
            'train_accuracy_final': train_accuracies[-1] if train_accuracies else 0,
            'train_loss_final': train_losses[-1] if train_losses else 0,
            'best_accuracy': best_accuracy,
            'epochs': epochs,
            'learning_rate': 5e-5,
            'best_model_path': best_model_path,
            'label_encoder_path': label_encoder_path,
            'metadata_path': metadata_path,
            'confidence_mean': np.mean(all_confidences),
            'confidence_std': np.std(all_confidences),
            'num_classes': num_classes
        }
        
        all_results.append(seed_results)
        all_predictions[seed] = {
            'preds': all_preds,
            'labels': all_labels,
            'probs': np.vstack(all_probs) if all_probs else np.array([])
        }
        
        print(f"\n  Final Results:")
        print(f"    Test Accuracy: {test_accuracy:.4f}")
        print(f"    Macro F1: {macro_f1:.4f}")
        print(f"    Weighted F1: {weighted_f1:.4f}")
        print(f"    Best Accuracy: {best_accuracy:.4f}")
        print(f"    Confidence: {np.mean(all_confidences):.4f} ± {np.std(all_confidences):.4f}")
        print(f"    Number of authors: {num_classes}")
        
        if seed == SEEDS[0]:
            print(f"\n  Performance on top 5 most frequent authors in test set:")
            
            label_counts = Counter(all_labels)
            top_5_indices = [idx for idx, _ in label_counts.most_common(5)]
            
            for i, idx in enumerate(top_5_indices):
                if idx < len(label_encoder.classes_):
                    author_name = label_encoder.inverse_transform([idx])[0]
                else:
                    author_name = f"Author_{idx}"
                
                author_mask = np.array(all_labels) == idx
                if author_mask.any():
                    author_acc = accuracy_score(
                        np.array(all_labels)[author_mask],
                        np.array(all_preds)[author_mask]
                    )
                    count = author_mask.sum()
                    print(f"    {author_name}: Accuracy = {author_acc:.4f} ({count} samples)")
        
        del model, optimizer, train_loader, test_loader
        torch.cuda.empty_cache()
        gc.collect()
    
    print("\n" + "="*80)
    print("COMPREHENSIVE RESULTS ANALYSIS")
    print("="*80)
    
    results_df = pd.DataFrame(all_results)
    
    print("\nPerformance Across Seeds:")
    print("-" * 60)
    
    metrics_to_show = ['test_accuracy', 'macro_f1', 'weighted_f1', 'train_accuracy_final', 'confidence_mean']
    
    for metric in metrics_to_show:
        mean_val = results_df[metric].mean()
        std_val = results_df[metric].std()
        min_val = results_df[metric].min()
        max_val = results_df[metric].max()
        
        metric_name = metric.replace('_', ' ').title()
        if metric == 'confidence_mean':
            metric_name = 'Average Confidence'
        
        print(f"{metric_name:25s}: {mean_val:.4f} ± {std_val:.4f}")
        print(f"  Range: [{min_val:.4f}, {max_val:.4f}]")
        print()
    
    print("\n" + "-" * 60)
    print("STATISTICAL ANALYSIS")
    print("-" * 60)
    
    if len(results_df) > 1:
        accuracies = results_df['test_accuracy']
        
        cv = (accuracies.std() / accuracies.mean()) * 100
        
        if 'num_classes' in results_df.columns:
            num_classes = int(results_df['num_classes'].iloc[0])
        else:
            num_classes = 1000
        
        random_chance = 1 / num_classes
        
        t_stat, p_value = stats.ttest_1samp(accuracies, random_chance)
        
        print(f"Accuracy Consistency:")
        print(f"  Mean: {accuracies.mean():.4f}")
        print(f"  Std: {accuracies.std():.4f}")
        print(f"  Coefficient of Variation: {cv:.2f}%")
        print(f"  Range: [{accuracies.min():.4f}, {accuracies.max():.4f}]")
        print()
        print(f"Statistical Significance vs Random Chance ({random_chance:.6f}):")
        print(f"  t-statistic: {t_stat:.4f}")
        print(f"  p-value: {p_value:.6f}")
        
        if p_value < 0.05:
            print("  ✓ Significantly better than random chance (p < 0.05)")
            improvement = (accuracies.mean() - random_chance) / random_chance * 100
            print(f"  ✓ Improvement over random: {improvement:.1f}%")
        else:
            print("  ✗ Not significantly better than random chance")
    
    print("\n" + "-" * 60)
    print("SAVING RESULTS")
    print("-" * 60)
    
    results_path = f"/home/aman_swaraj/Downloads/Codelite/codellama_{model_size}_authorship_results.csv"
    results_df.to_csv(results_path, index=False)
    print(f"Detailed results saved to: {results_path}")
    
    predictions_path = f"/home/aman_swaraj/Downloads/Codelite/codellama_{model_size}_predictions.pkl"
    import pickle
    with open(predictions_path, 'wb') as f:
        pickle.dump(all_predictions, f)
    print(f"Predictions saved to: {predictions_path}")
    
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        
        print("\nGenerating visualizations...")
        
        plt.figure(figsize=(12, 6))
        
        seeds = [f"Seed {s}" for s in results_df['seed']]
        x_pos = np.arange(len(seeds))
        
        bars = plt.bar(x_pos, results_df['test_accuracy'], color='#6A0DAD', alpha=0.8)
        plt.axhline(y=results_df['test_accuracy'].mean(), color='red', 
                   linestyle='--', linewidth=2, label=f'Mean: {results_df["test_accuracy"].mean():.4f}')
        plt.xlabel('Random Seed')
        plt.ylabel('Test Accuracy')
        plt.title(f'CodeLlama-{model_size}: Authorship Attribution Across {num_seeds} Seeds')
        plt.xticks(x_pos, seeds)
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        for bar, acc in zip(bars, results_df['test_accuracy']):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{acc:.4f}', ha='center', va='bottom', fontsize=9)
        
        plt.tight_layout()
        plt.savefig(f'codellama_{model_size}_authorship_accuracy.png', dpi=300, bbox_inches='tight')
        print(f"  Saved: codellama_{model_size}_authorship_accuracy.png")
        
        plt.figure(figsize=(10, 6))
        
        metrics = ['test_accuracy', 'macro_f1', 'weighted_f1']
        colors = ['#6A0DAD', '#9D4EDD', '#C77DFF']
        
        x = np.arange(len(results_df))
        width = 0.25
        
        for idx, (metric, color) in enumerate(zip(metrics, colors)):
            offset = (idx - 1) * width
            plt.bar(x + offset, results_df[metric], width, 
                   color=color, alpha=0.8, label=metric.replace('_', ' ').title())
        
        plt.xlabel('Seed')
        plt.ylabel('Score')
        plt.title(f'CodeLlama-{model_size}: Performance Metrics Comparison')
        plt.xticks(x, seeds)
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'codellama_{model_size}_authorship_metrics.png', dpi=300, bbox_inches='tight')
        print(f"  Saved: codellama_{model_size}_authorship_metrics.png")
        
        plt.show()
        
    except ImportError:
        print("\nVisualization libraries not available. Skipping plots.")

if __name__ == "__main__":
    print("CodeLlama Authorship Attribution Experiment")
    print("="*80)
    
    if not torch.cuda.is_available():
        print("WARNING: No GPU detected. CodeLlama requires GPU for reasonable training time.")
        print("Continue anyway? (y/n)")
        choice = input().strip().lower()
        if choice != 'y':
            exit()
    
    print("\nSelect CodeLlama model size:")
    print("7b   - 7 billion parameters (default, fits in 16GB GPU)")
    print("13b  - 13 billion parameters (needs ~24GB GPU)")
    print("34b  - 34 billion parameters (needs ~48GB GPU)")
    print("="*40)
    
    model_size = input("Enter model size (default: 7b): ").strip() or "7b"
    
    if model_size not in ["7b", "13b", "34b"]:
        print(f"Invalid model size. Using 7b.")
        model_size = "7b"
    
    print(f"\nSelect number of seeds to evaluate:")
    print("1 - Quick test (fastest)")
    print("3 - Good balance (recommended)")
    print("5 - Most robust (slowest)")
    print("="*40)
    
    try:
        num_seeds = int(input("Enter number of seeds (default: 3): ").strip() or "10")
        num_seeds = max(1, min(10, num_seeds))  
    except:
        num_seeds = 3
    
    print(f"\nSelect number of training epochs:")
    print("3 - Quick training")
    print("6 - Standard training (default, based on previous success)")
    print("10 - Extended training")
    print("="*40)
    
    try:
        num_epochs = int(input("Enter number of epochs (default: 6): ").strip() or "6")
        num_epochs = max(1, min(20, num_epochs))  
    except:
        num_epochs = 6
    
    print(f"\nStarting experiment with:")
    print(f"  Model: CodeLlama-{model_size}")
    print(f"  Seeds: {num_seeds}")
    print(f"  Epochs: {num_epochs}")
    print(f"  Dataset: Google Code Jam 2008")
    print("="*80)
    
    results_df = run_codellama_experiment(num_seeds=num_seeds, model_size=model_size, num_epochs=num_epochs)
    
    print("\n" + "="*80)
    print("FINAL SUMMARY")
    print("="*80)
    
    if not results_df.empty:
        print(f"Model: CodeLlama-{model_size}")
        print(f"Number of seeds evaluated: {len(results_df)}")
        
        if 'num_classes' in results_df.columns:
            num_authors = int(results_df['num_classes'].iloc[0])
        else:
            num_authors = "unknown"
            
        print(f"Number of authors: {num_authors}")
        print(f"Average test accuracy: {results_df['test_accuracy'].mean():.4f}")
        print(f"Accuracy range: [{results_df['test_accuracy'].min():.4f}, {results_df['test_accuracy'].max():.4f}]")
        
        if len(results_df) > 1:
            print(f"Best seed: {int(results_df.loc[results_df['test_accuracy'].idxmax(), 'seed'])}")
            print(f"Worst seed: {int(results_df.loc[results_df['test_accuracy'].idxmin(), 'seed'])}")
        
        final_summary = {
            'model': f'CodeLlama-{model_size}',
            'num_seeds': len(results_df),
            'num_authors': num_authors,
            'mean_accuracy': results_df['test_accuracy'].mean(),
            'std_accuracy': results_df['test_accuracy'].std(),
            'mean_macro_f1': results_df['macro_f1'].mean(),
            'std_macro_f1': results_df['macro_f1'].std(),
            'best_accuracy': results_df['test_accuracy'].max(),
            'worst_accuracy': results_df['test_accuracy'].min(),
            'confidence_mean': results_df['confidence_mean'].mean()
        }
        
        summary_path = f"/home/aman_swaraj/Downloads/Codelite/codellama_{model_size}_final_summary.csv"
        pd.DataFrame([final_summary]).to_csv(summary_path, index=False)
        print(f"\nFinal summary saved to: {summary_path}")
        
        if results_df['test_accuracy'].mean() >= 0.85:
            print("\n" + "="*80)
            print("🎉 OUTSTANDING ACHIEVEMENT!")
            print("="*80)
            print(f"CodeLlama-{model_size} achieved {results_df['test_accuracy'].mean():.2%} accuracy")
            print(f"on authorship attribution across {num_authors} authors!")
            print("\nThis represents a breakthrough in code authorship attribution.")
            print("For comparison:")
            print("  • Random chance: {:.4f}%".format(100/num_authors if isinstance(num_authors, int) else "unknown"))
            print("  • State-of-the-art typically: 70-85%")
            print("  • Your result: {:.2f}%".format(results_df['test_accuracy'].mean()*100))
    else:
        print("No results were generated. Please check for errors.")